# Phase 1: Data Acquisition and Initial Exploration

**Course**: KTH SF2943 Time Series Analysis
**Data source**: ENTSO-E Transparency Platform
**URL**: https://transparency.entsoe.eu
**Navigation**: Consumption → Actual Total Load → Bidding Zone: AT
**Column used**: "Actual Total Load [MW]" (stored here as `load_MWh`)
**Download date**: April 2025

## Design: Dual-Dataset Parallel Comparison

The full ARMA pipeline is executed **in parallel on two independent data sources**.
Both datasets cover Austria hourly electricity load; results from both are compared
to assess whether conclusions depend on the data source.

| | Dataset A | Dataset B |
|---|---|---|
| Source | Two ENTSO-E CSVs (2015–2025) | Legacy `data_hourly.csv` |
| Coverage | ~96k hourly obs | ~50k hourly obs |
| Split | 80:20 chronological | 80:20 chronological |
| Variables | `X_train`, `X_test` | `X_train_B`, `X_test_B` |

In [ ]:
%matplotlib inline
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.stattools import acf

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.size": 11,
})
os.makedirs("output", exist_ok=True)
print("Imports OK")

## 1. Load Data

### Dataset A — Combined ENTSOE CSVs

Both files share schema `timestamp_local, load_MWh` with mixed UTC offsets
(+01:00 CET / +02:00 CEST). Normalise to UTC before concat to preserve a
proper `DatetimeIndex`, then convert to `Europe/Vienna` for display.

In [ ]:
def load_and_utc(path):
    df = pd.read_csv(path, parse_dates=["timestamp_local"], index_col="timestamp_local")
    df.index = pd.to_datetime(df.index, utc=True)
    return df

df1 = load_and_utc("austria_hourly_load_2015_2019_MWh.csv")
df2 = load_and_utc("austria_hourly_load_2020_2025_MWh.csv")

combined = pd.concat([df1, df2]).sort_index()
combined.index = combined.index.tz_convert("Europe/Vienna")
X_all_A = combined["load_MWh"]

print(f"df1 rows  : {len(df1):,}  ({df1.index[0]} → {df1.index[-1]})")
print(f"df2 rows  : {len(df2):,}  ({df2.index[0]} → {df2.index[-1]})")
print(f"Combined  : {len(X_all_A):,} rows  |  tz: {X_all_A.index.tz}")

### Dataset B — Legacy ENTSOE Export (`data_hourly.csv`)

Semicolon-separated, 6-row multi-level header. `pd.to_numeric(errors="coerce")`
skips the URL/unit metadata rows that bleed past the 3-level header.

In [ ]:
df_b = pd.read_csv("data_hourly.csv", sep=";",
                   header=[0, 1, 2], index_col=0, parse_dates=True)
X_all_B = pd.to_numeric(df_b["AT"]["load"]["actual_entsoe_transparency"],
                         errors="coerce").dropna()
X_all_B.index = pd.to_datetime(X_all_B.index, utc=True).tz_convert("Europe/Vienna")
X_all_B.index.name = "timestamp_local"

print(f"Dataset B : {len(X_all_B):,} rows  ({X_all_B.index[0]} → {X_all_B.index[-1]})")

## 2. Quality Check — DST Gaps & Duplicates

ENTSO-E data has structural gaps from DST transitions:
- **Spring forward** (last Sunday March): 1 h skipped per year → NaN after `asfreq("h")`.
- **Fall back** (last Sunday October): 1 h repeated → duplicate timestamp.

Strategy: drop duplicates (keep first), fill NaNs with linear interpolation.
Applied **per source file** (Dataset A) to avoid masking the boundary gap between df1/df2.

In [ ]:
def dst_clean(s, label):
    """Dedup + hourly reindex + linear interpolation. Returns cleaned series."""
    s = s[~s.index.duplicated(keep="first")]
    s_h = s.asfreq("h")
    n_gap = int(s_h.isna().sum())
    s_h = s_h.interpolate(method="linear")
    print(f"{label}: {len(s):,} raw → {len(s_h):,} hourly | gaps filled: {n_gap} | NaN remaining: {s_h.isna().sum()}")
    return s_h

# Dataset A — per source file
s1 = df1["load_MWh"].copy()
s1.index = s1.index.tz_convert("Europe/Vienna")
s2 = df2["load_MWh"].copy()
s2.index = s2.index.tz_convert("Europe/Vienna")
s1_clean = dst_clean(s1, "A-file1 (2015-2019)")
s2_clean = dst_clean(s2, "A-file2 (2020-2025)")

# Dataset B — single file
X_all_B_q = X_all_B[~X_all_B.index.duplicated(keep="first")]
X_all_B_q = X_all_B_q.asfreq("h")
n_gap_b = int(X_all_B_q.isna().sum())
X_all_B_q = X_all_B_q.interpolate(method="linear")
print(f"B-file    : {len(X_all_B):,} raw → {len(X_all_B_q):,} hourly | gaps filled: {n_gap_b} | NaN remaining: {X_all_B_q.isna().sum()}")

# Combine Dataset A files after per-file DST cleaning
X_all_A_q = pd.concat([s1_clean, s2_clean]).sort_index()
X_all_A_q = X_all_A_q[~X_all_A_q.index.duplicated(keep="first")]
X_all_A_q = X_all_A_q.asfreq("h").interpolate(method="linear")
print(f"Dataset A combined: {len(X_all_A_q):,} hourly obs")

## 3. Raw Series Visualisation

### Plot 1: Full Raw Series (Both Datasets)

Pre-treatment view of the complete series. Expected features: stable long-run level,
winter peaks (Dec–Feb), summer valleys (Jul–Aug), and dense vertical striping from
intraday cycles. The 2016 data artefact (near-zero readings) is visible here and will
be addressed in the next section.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), constrained_layout=True)

for ax, s, label, color in zip(
    axes,
    [X_all_A_q, X_all_B_q],
    ["Dataset A", "Dataset B"],
    ["steelblue", "darkgreen"],
):
    ax.plot(s.index, s.values, linewidth=0.35, color=color)
    ax.set_title(
        f"{label} — Raw Series  "
        f"({s.index[0].year}–{s.index[-1].year}, n={len(s):,})",
        fontsize=12, fontweight="bold",
    )
    ax.set_xlabel("Date", fontsize=11)
    ax.set_ylabel("Load [MWh/h]", fontsize=11)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle("Austria Hourly Electricity Load — Raw Series (Dual Dataset)",
             fontsize=13, fontweight="bold")
plt.savefig("output/phase1_load_raw.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/phase1_load_raw.png")

## 4. Outlier Detection and Treatment

Two classes of artefact are present in the raw series and require different treatments.

**Sub-physical readings** (< 2 000 MWh/h): isolated near-zero values arising from the
ENTSO-E portal transition at end-2015, where APG submitted uncorrected placeholder readings.

**Documented multi-hour artefact windows**: the ENTSO-E transition corrupted three
complete days in 2015 — 29 November, 25 December, and 29 December — where the TSO
submitted systematically low placeholder readings across all 24 hours of each day
(cross-year minima: 664, 1180, and 1627 MWh/h respectively, against a peer-year
floor of ≥ 4 859 MWh/h for the same calendar days in 2016–2019).
These windows coincide with a documented data-collection anomaly (W3 Step 1; B&D §1.5,
p. 20) and span too many consecutive hours for linear interpolation to recover the diurnal
cycle.

**Treatment**: sub-physical isolated points are replaced by linear interpolation between
surrounding clean values. Multi-hour artefact windows are replaced by the corresponding
value from the same clock-hour seven days earlier — the adjacent period of the same
seasonal phase, as prescribed in B&D §1.5, p. 20. Using the prior-week same-hour
preserves the diurnal and weekly seasonality that a straight-line fill through a 24-hour
gap cannot recover. Artefact points are marked in red on the plot below.

In [ ]:
PHY_MIN = 2_000  # MWh/h — instrument-error floor

# Documented multi-hour artefact windows (ENTSO-E portal transition, 2015)
# Three complete days confirmed via cross-year comparison (2016-2019 same calendar day)
ARTEFACT_WINDOWS_A = [
    ("2015-11-29 00:00", "2015-11-29 23:00"),  # full Sunday artefact (min 664 vs peer 6017)
    ("2015-12-25 00:00", "2015-12-25 23:00"),  # full Christmas Day artefact (min 1180 vs peer 4859)
    ("2015-12-29 00:00", "2015-12-29 23:00"),  # full day artefact (min 1627 vs peer 5539)
]

def build_mask(s):
    """Sub-physical flag only (PHY_MIN threshold)."""
    return (s < PHY_MIN).astype(bool)

def seasonal_phase_fill(s, windows, lag_hours=168):
    """Replace artefact windows with same clock-hour lag_hours (7 days) earlier."""
    out = s.copy()
    for start, end in windows:
        idx = s.loc[start:end].index
        for ts in idx:
            ref = ts - pd.Timedelta(hours=lag_hours)
            if ref in s.index and not pd.isna(s[ref]):
                out[ts] = s[ref]
    return out

mask_A = build_mask(X_all_A_q)
mask_B = build_mask(X_all_B_q)

# Step 1: mark all artefact windows on Dataset A for plotting
mask_A_windows = mask_A.copy()
for start, end in ARTEFACT_WINDOWS_A:
    mask_A_windows.loc[start:end] = True

fig, axes = plt.subplots(2, 1, figsize=(16, 8), constrained_layout=True)
for ax, s, mask, label, color in zip(
    axes,
    [X_all_A_q, X_all_B_q],
    [mask_A_windows, mask_B],
    ["Dataset A", "Dataset B"],
    ["steelblue", "darkgreen"],
):
    ax.plot(s.index, s.values, color=color, lw=0.35, label="Load", zorder=1)
    ax.scatter(s.index[mask], s.values[mask],
               color="red", s=18, zorder=5, label=f"Artefact ({int(mask.sum())} pts)")
    ax.set_title(f"{label} — Outlier Detection", fontsize=11, fontweight="bold")
    ax.set_xlabel("Date", fontsize=11)
    ax.set_ylabel("Load [MWh/h]", fontsize=11)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(fontsize=10)
fig.suptitle("Outlier Detection — Austria Hourly Load (Both Datasets)",
             fontsize=13, fontweight="bold")
plt.savefig("output/phase1_outlier_detection.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/phase1_outlier_detection.png")

# Step 2: treat Dataset A — seasonal phase fill for documented windows, then linear for isolated sub-physical
X_A_phase_filled = seasonal_phase_fill(X_all_A_q, ARTEFACT_WINDOWS_A, lag_hours=168)
X_all_A_final = X_A_phase_filled.where(~mask_A).interpolate(method="linear")

# Dataset B — only sub-physical linear interpolation (no documented window artefact)
X_all_B_final = X_all_B_q.where(~mask_B).interpolate(method="linear")

n_win_A = sum(len(X_all_A_q.loc[s:e]) for s, e in ARTEFACT_WINDOWS_A)
print(f"Dataset A: {int(mask_A.sum())} sub-physical + {n_win_A} window-artefact hours replaced")
print(f"Dataset B: {int(mask_B.sum())} sub-physical hours replaced")

### Post-Treatment Series

Full series after linear interpolation of anomalous points. The 2016 artefact dip
should no longer be visible; all other values are unchanged.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), constrained_layout=True)

for ax, s, label, color in zip(
    axes,
    [X_all_A_final, X_all_B_final],
    ["Dataset A", "Dataset B"],
    ["steelblue", "darkgreen"],
):
    ax.plot(s.index, s.values, linewidth=0.35, color=color)
    ax.set_title(
        f"{label} — Post-Treatment  "
        f"({s.index[0].year}–{s.index[-1].year}, n={len(s):,})",
        fontsize=12, fontweight="bold",
    )
    ax.set_xlabel("Date", fontsize=11)
    ax.set_ylabel("Load [MWh/h]", fontsize=11)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle("Austria Hourly Electricity Load — After Outlier Treatment (Dual Dataset)",
             fontsize=13, fontweight="bold")
plt.savefig("output/phase1_load_treated.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/phase1_load_treated.png")

## 5. Train / Test Split — 80:20 Chronological

An 80:20 chronological split is applied **independently** to each dataset.
The split index is `floor(0.8 × N)` so neither dataset influences the other's boundary.

> All decomposition parameters (trend, seasonal indices) will be estimated on
> the training partition only — no look-ahead into the test set.

In [ ]:
n_A      = len(X_all_A_final)
split_A  = int(0.8 * n_A)
X_train  = X_all_A_final.iloc[:split_A]
X_test   = X_all_A_final.iloc[split_A:]

n_B      = len(X_all_B_final)
split_B  = int(0.8 * n_B)
X_train_B = X_all_B_final.iloc[:split_B]
X_test_B  = X_all_B_final.iloc[split_B:]

print(f"{'':12s} {'N total':>10s}  {'N train':>10s}  {'Train start':>24s}  {'Train end':>24s}  {'N test':>8s}")
print(f"{'Dataset A':12s} {n_A:>10,}  {len(X_train):>10,}  {str(X_train.index[0]):>24s}  {str(X_train.index[-1]):>24s}  {len(X_test):>8,}")
print(f"{'Dataset B':12s} {n_B:>10,}  {len(X_train_B):>10,}  {str(X_train_B.index[0]):>24s}  {str(X_train_B.index[-1]):>24s}  {len(X_test_B):>8,}")
print()
print("Descriptive statistics — training partitions:")
print(f"{'':20s} {'Dataset A':>12s}  {'Dataset B':>12s}")
for label, fa, fb in [
    ("Mean [MWh/h]",  X_train.mean(),  X_train_B.mean()),
    ("Std  [MWh/h]",  X_train.std(),   X_train_B.std()),
    ("Min  [MWh/h]",  X_train.min(),   X_train_B.min()),
    ("Max  [MWh/h]",  X_train.max(),   X_train_B.max()),
]:
    print(f"  {label:18s} {fa:>12.1f}  {fb:>12.1f}")

## 6. Seasonality Zoom — Clean Training Data

### Plot 2: One-Week Zoom — Intraday Seasonality (Both Datasets)

Week of 12–18 January 2015 (winter, high load), plotted on outlier-corrected training data.
Both datasets should show the same 24-h intraday cycle. Divergence in this period would
indicate source-specific issues.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), constrained_layout=True)

for ax, s, label, color in zip(
    axes,
    [X_train, X_train_B],
    ["Dataset A", "Dataset B"],
    ["darkblue", "darkgreen"],
):
    ws = pd.Timestamp("2015-01-12", tz=s.index.tz)
    we = ws + pd.Timedelta(weeks=1)
    sw = s.loc[ws:we]
    ax.plot(sw.index, sw.values, linewidth=1.5,
            marker="o", markersize=3, color=color)
    for day in pd.date_range(ws, we, freq="D", tz=s.index.tz)[:-1]:
        ax.axvline(day, color="gray", linestyle=":", alpha=0.6, linewidth=0.8)
    ax.set_title(f"{label} — Jan 12–18, 2015  ({len(sw)} points)",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Time [CET]", fontsize=11)
    ax.set_ylabel("Load [MWh/h]", fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())

fig.suptitle("Intraday Seasonality — 1-Week Zoom (Both Datasets)", fontsize=13, fontweight="bold")
fig.autofmt_xdate(rotation=30)
plt.savefig("output/phase1_load_1week.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/phase1_load_1week.png")

### Plot 3: One-Year Zoom — Annual Seasonality 2017 (Both Datasets)

Full calendar year at hourly resolution on clean training data. The year shows winter
peaks, a summer trough, and intermediate shoulders. Year 2017 falls within the training
partition of both datasets.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), constrained_layout=True)

for ax, s, label, color in zip(
    axes,
    [X_train.loc["2017"], X_train_B.loc["2017"]],
    ["Dataset A", "Dataset B"],
    ["darkgreen", "darkorange"],
):
    ax.plot(s.index, s.values, linewidth=0.45, color=color, alpha=0.85)
    win = s.loc["2017-01":"2017-02"].mean()
    sum_ = s.loc["2017-07":"2017-08"].mean()
    ax.set_title(
        f"{label} — 2017  |  Winter mean: {win:.0f}  Summer mean: {sum_:.0f}  "
        f"Amplitude: {win - sum_:.0f} MWh/h",
        fontsize=11, fontweight="bold",
    )
    ax.set_xlabel("Month", fontsize=11)
    ax.set_ylabel("Load [MWh/h]", fontsize=11)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

fig.suptitle("Annual Seasonality — 2017 (Both Datasets)", fontsize=13, fontweight="bold")
plt.savefig("output/phase1_load_1year.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/phase1_load_1year.png")

### Plot 4: Two-Year Zoom — Annual Seasonality 2016–2017 (Both Datasets)

Over two full calendar years, the annual cycle is visible twice, with two consecutive
winter peaks and summer troughs. Year 2016 covers the post-artefact recovery period;
no residual dip should appear in the treated series.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), constrained_layout=True)

for ax, s, label, color in zip(
    axes,
    [X_train.loc["2016":"2017"], X_train_B.loc["2016":"2017"]],
    ["Dataset A", "Dataset B"],
    ["darkblue", "darkorange"],
):
    ax.plot(s.index, s.values, linewidth=0.4, color=color, alpha=0.85)
    ax.set_title(
        f"{label} — 2016–2017  (n={len(s):,})",
        fontsize=12, fontweight="bold",
    )
    ax.set_xlabel("Month", fontsize=11)
    ax.set_ylabel("Load [MWh/h]", fontsize=11)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

fig.suptitle("Annual Seasonality — 2-Year Zoom 2016–2017 (Both Datasets)",
             fontsize=13, fontweight="bold")
fig.autofmt_xdate(rotation=30)
plt.savefig("output/phase1_load_2year.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/phase1_load_2year.png")

## 7. Sample ACF — up to Lag 500 (Both Datasets)

$$\hat{\rho}(h) = \frac{\hat{\gamma}(h)}{\hat{\gamma}(0)}, \quad
\hat{\gamma}(h) = \frac{1}{n}\sum_{t=1}^{n-h}(X_{t+h}-\bar{X})(X_t-\bar{X})$$

95% CI: $\pm 1.96/\sqrt{n}$ (valid under white-noise null hypothesis).

> `fft=True` applies the Wiener-Khinchin theorem: $O(n \log n)$ vs $O(n^2)$.

In [ ]:
N_LAGS = 500

acf_A = acf(X_train.values,   nlags=N_LAGS, fft=True, alpha=None)
acf_B = acf(X_train_B.values, nlags=N_LAGS, fft=True, alpha=None)
lags  = np.arange(N_LAGS + 1)

ci_A = 1.96 / np.sqrt(len(X_train))
ci_B = 1.96 / np.sqrt(len(X_train_B))

fig, axes = plt.subplots(2, 1, figsize=(16, 9), constrained_layout=True)

for ax, vals, ci, n, label, color in zip(
    axes,
    [acf_A, acf_B],
    [ci_A, ci_B],
    [len(X_train), len(X_train_B)],
    ["Dataset A", "Dataset B"],
    ["steelblue", "darkgreen"],
):
    ax.bar(lags, vals, width=0.8, color=color, alpha=0.7)
    ax.axhline( ci, color="red", linestyle="--", linewidth=1, label=f"95% CI: ±{ci:.4f}")
    ax.axhline(-ci, color="red", linestyle="--", linewidth=1)
    ax.axhline(0, color="black", linewidth=0.4)
    for h in [24, 48, 168, 336]:
        ax.axvline(h, color="orange", linestyle=":", alpha=0.75, linewidth=1)
        ax.text(h + 2, vals[h] + 0.02, f"h={h}", fontsize=7.5, color="darkorange", va="bottom")
    ax.set_title(f"Sample ACF — {label}  (n={n:,}, lags 0–{N_LAGS})",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Lag [hours]", fontsize=11)
    ax.set_ylabel("ACF", fontsize=11)
    ax.set_xlim(-2, N_LAGS + 2)
    ax.legend(fontsize=10)

fig.suptitle("Sample ACF — Austria Hourly Load, Both Datasets", fontsize=13, fontweight="bold")
plt.savefig("output/phase1_acf_raw.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'Lag':>6s}  {'ACF-A':>10s}  {'ACF-B':>10s}")
for h in [1, 24, 48, 168, 336, 500]:
    print(f"{h:>6d}  {acf_A[h]:>10.4f}  {acf_B[h]:>10.4f}")
print("Saved: output/phase1_acf_raw.png")

### ACF Interpretation

1. **Slow, non-decaying envelope** in both datasets: ACF stays far above the 95% CI throughout
   all 500 lags — the hallmark of a **non-stationary** series. Applying ARMA directly is invalid.

2. **Sharp spikes at $h = 24, 48, 72, \ldots$**: Intraday seasonal period $d_3 = 24$ h.

3. **Prominent spikes at $h = 168, 336, \ldots$**: Weekly seasonal period $d_2 = 168$ h.

4. **Annual spike at $h = 8760$** (outside the lag-500 window): $d_1 = 8760$ h. The
   recurring winter peaks in Plots 3 and 4 are consistent with this period.

5. **Both datasets agree**: the ACF structure is consistent across both sources; modelling
   conclusions are therefore independent of the data source.

6. **Conclusion**: Triple seasonality ($d = 24, 168, 8760$). Phase 2 will remove trend $m_t$
   and three seasonal components to obtain a stationary residual $Z_t$.

## Phase 1 Summary

| Item | Dataset A | Dataset B |
|------|-----------|-----------|
| Source | Two ENTSO-E CSVs | Legacy `data_hourly.csv` |
| Outlier treatment | Linear interp on points < 2,000 MWh/h | Linear interp on points < 2,000 MWh/h |
| Train N | ~77,126 | ~40,320 |
| Test N | ~19,282 | ~10,080 |
| Raw series stationary? | **No** — ACF non-decaying | **No** — same pattern |
| Seasonal periods | $d=24, 168, 8760$ | $d=24, 168, 8760$ |
| Outputs | `output/phase1_*.png` | same folder |

## Data Export

Export the cleaned Dataset A series to `data/` so downstream phases load a pre-processed
series rather than re-applying outlier treatment independently.

In [ ]:
import os
os.makedirs("data", exist_ok=True)

export_path = "data/austria_load_A_cleaned.csv"
X_all_A_final.to_frame(name="load_MWh").to_csv(export_path)
print(f"Exported cleaned Dataset A: {len(X_all_A_final):,} obs → {export_path}")
print(f"  Range: {X_all_A_final.index[0].date()} → {X_all_A_final.index[-1].date()}")
print(f"  Min: {X_all_A_final.min():.1f} MWh/h  |  Max: {X_all_A_final.max():.1f} MWh/h")
print(f"  NaN remaining: {X_all_A_final.isna().sum()}")